# Assignment 27 Iris deployment study
Tasks 1–4 adapted from retail image recognition to classifying three Iris species from four measurements. This is an educational prototype for a field botany application, not a validated general plant identifier. Two deep multilayer perceptrons replace image CNNs because the inputs are tabular measurements. Run all cells in order. Outputs below were generated by executing this notebook.

## Task 1 Compare two architectures
**Larger MLP 4 → 64 → 32 → 3:** two ReLU hidden layers learn nonlinear combinations; softmax returns three class probabilities. It has 2,499 parameters. More capacity may help complex relationships, but is excessive for 150 samples, costs more memory and may overfit.

**Compact MLP 4 → 16 → 8 → 3:** same feedforward design with 243 parameters. It needs fewer multiply operations and less parameter storage; lower capacity may underfit harder datasets. Both require standardisation fitted only on training data. Neither uses convolution, image features, residual connections or spatial invariance.

**Application:** a field worker enters sepal length, sepal width, petal length and petal width in centimetres to classify setosa, versicolor or virginica offline. The compact network suits a device with limited memory; the larger network is a comparison baseline. Iris has only three species and cannot support identification of arbitrary plants. A separate image dataset would be required for a camera-based application.

In [1]:
# If dependencies are missing, uncomment and run this line once, then restart the kernel.
# %pip install mlflow scikit-learn pandas numpy joblib cloudpickle
from pathlib import Path
from time import perf_counter
import copy, hashlib, json, platform
import numpy as np
import pandas as pd
import joblib, cloudpickle, sklearn, mlflow
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from mlflow import MlflowClient
from mlflow.models import infer_signature

SEED = 42
features = ['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm']
iris = load_iris()
# Self-contained: use the bundled Iris dataset; no CSV download is required.
X = pd.DataFrame(iris.data, columns=features)
y = np.array(['Iris-' + iris.target_names[i] for i in iris.target])
indices = np.arange(len(X))
train_idx, rest_idx = train_test_split(indices, test_size=0.4, stratify=y, random_state=SEED)
valid_idx, test_idx = train_test_split(rest_idx, test_size=0.5, stratify=y[rest_idx], random_state=SEED)
scaler = StandardScaler().fit(X.iloc[train_idx])
Z = scaler.transform(X)
output = Path('iris_assignment_outputs'); output.mkdir(exist_ok=True)
mlflow.set_tracking_uri('sqlite:///' + str((output / 'iris_mlflow.db').resolve()))
experiment = mlflow.set_experiment('Assignment27_Iris_Neural_Deployment')
manifest = {'train':train_idx.tolist(), 'validation':valid_idx.tolist(), 'test':test_idx.tolist()}
dataset_hash = hashlib.sha256(X.to_csv(index=False).encode() + y.tobytes()).hexdigest()
print('Data:', X.shape, '| classes:', np.unique(y))
print('Split:',len(train_idx),'training,',len(valid_idx),'validation,',len(test_idx),'test')
print('MLflow tracking:', mlflow.get_tracking_uri())


2026/10/15 13:21:58 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /root/.local/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.
Data: (150, 4) | classes: ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']
Split: 90 training, 30 validation, 30 test
MLflow tracking: sqlite:////workspace/scratch/e982596c1aea/assignment27/iris_assignment_outputs/iris_mlflow.db


In [2]:
models = {}
training_ms = {}
for name, widths in [('large_mlp',(64,32)),('compact_mlp',(16,8))]:
    model = MLPClassifier(hidden_layer_sizes=widths, activation='relu', solver='lbfgs',
                          alpha=0.01, max_iter=3000, random_state=SEED)
    start = perf_counter()
    model.fit(Z[train_idx], y[train_idx])
    training_ms[name] = (perf_counter()-start)*1000
    models[name] = model
    count = sum(a.size for a in model.coefs_ + model.intercepts_)
    print(name, 'parameters:',count,'iterations:',model.n_iter_)


large_mlp parameters: 2499 iterations: 103
compact_mlp parameters: 243 iterations: 113


## Task 2 Weight quantisation
I recommend symmetric per-layer 8-bit weight quantisation of the compact network. For each weight matrix W, scale s = max(abs(W))/127; store q = round(W/s) clipped to [−127,127] as int8. Approximate weights during inference by q × s. Biases, scales and preprocessing stay floating point; this is weight-only quantisation, not full integer inference.

Weight storage falls from 8 bytes per float64 weight to 1 byte per int8 weight, plus one scale per layer. Whole-file savings are smaller because metadata and biases remain. Rounding can change decision boundaries, so validation accuracy and macro F1 must be compared with the original network. This portable NumPy demonstration reconstructs weights for computation: it does not guarantee faster inference. An optimised integer runtime and actual target-device measurements are needed before claiming speed, energy or monetary savings. Offline operation removes the need for connectivity during prediction; quantisation reduces update payloads but does not improve Wi-Fi coverage.

In [3]:
class IrisRuntime:
    def __init__(self, network, scaler, quantized=False):
        self.classes = network.classes_.copy()
        self.mean = scaler.mean_.copy()
        self.std = scaler.scale_.copy()
        self.biases = [b.copy() for b in network.intercepts_]
        self.quantized = quantized
        self.weights = []
        self.scales = []
        for w in network.coefs_:
            scale = float(np.max(np.abs(w))/127) or 1.0
            self.weights.append(np.clip(np.rint(w/scale),-127,127).astype(np.int8) if quantized else w.copy())
            self.scales.append(scale if quantized else 1.0)
    def predict_proba(self, data):
        a = (np.asarray(data,dtype=float)-self.mean)/self.std
        for i,(w,b,s) in enumerate(zip(self.weights,self.biases,self.scales)):
            a = a @ (w.astype(float)*s if self.quantized else w) + b
            if i < len(self.weights)-1: a = np.maximum(a,0)
        a = np.exp(a-a.max(axis=1,keepdims=True))
        return a/a.sum(axis=1,keepdims=True)
    def predict(self, data):
        return self.classes[self.predict_proba(data).argmax(axis=1)]

runtimes = {k:IrisRuntime(v,scaler) for k,v in models.items()}
runtimes['compact_int8'] = IrisRuntime(models['compact_mlp'],scaler,True)
assert np.allclose(runtimes['compact_mlp'].predict_proba(X), models['compact_mlp'].predict_proba(Z))

def benchmark(runtime):
    sample = X.iloc[[valid_idx[0]]].to_numpy()
    for _ in range(30): runtime.predict(sample)
    times = []
    for _ in range(500):
        start = perf_counter(); runtime.predict(sample)
        times.append((perf_counter()-start)*1000)
    return float(np.percentile(times,50)),float(np.percentile(times,95))

rows = []
for name, runtime in runtimes.items():
    path = output / (name+'.pkl')
    path.write_bytes(cloudpickle.dumps(runtime))
    pred = runtime.predict(X.iloc[valid_idx])
    p50,p95 = benchmark(runtime)
    rows.append(dict(model=name,validation_accuracy=accuracy_score(y[valid_idx],pred),
        validation_macro_f1=f1_score(y[valid_idx],pred,average='macro'),
        model_size_bytes=path.stat().st_size,prediction_p50_ms=p50,prediction_p95_ms=p95,
        parameter_count=sum(w.size for w in runtime.weights)+sum(b.size for b in runtime.biases)))
comparison = pd.DataFrame(rows)
print(comparison.round(5).to_string(index=False))
comparison.to_csv(output/'comparison.csv',index=False)


       model  validation_accuracy  validation_macro_f1  model_size_bytes  prediction_p50_ms  prediction_p95_ms  parameter_count
   large_mlp              0.96667              0.96658             24092            0.01975            0.02378             2499
 compact_mlp              0.96667              0.96658              6044            0.01365            0.01410              243
compact_int8              0.96667              0.96658              4568            0.01643            0.01710              243


## Task 3 MLflow deployment support
Each run records architecture, quantisation, seed, split sizes, training settings, validation accuracy, macro F1, single-flower latency, serialized size and parameter count. The dataset, split indices, hash and software versions provide reproduction evidence. Both floating point networks and the quantised compact network use the same split and runtime wrapper.

The comparison queries only run IDs created in this execution. Each saved model is registered under IrisNeuralDeployment, creating a version linked to its run. Selection uses validation accuracy, then serialized size and p95 latency as tie breakers. The candidate alias points to the chosen version; a fixed version URI is used to reload it. Re-running adds versions, so no version number is hard-coded. Run IDs and output values are captured automatically. Versioning makes rollback possible, while pinned dependencies and feature order help reproduce inference.

In [4]:
class IrisPythonModel(mlflow.pyfunc.PythonModel):
    def __init__(self, runtime): self.runtime = runtime
    def predict(self, context, model_input, params=None):
        return self.runtime.predict(model_input)

client = MlflowClient()
run_records = {}
for row in rows:
    name = row['model']; runtime = runtimes[name]
    with mlflow.start_run(run_name=name) as run:
        mlflow.log_params({'architecture':str(models['large_mlp' if name=='large_mlp' else 'compact_mlp'].hidden_layer_sizes),
            'quantization':'int8_weights' if runtime.quantized else 'float64',
            'seed':SEED,'train_samples':90,'validation_samples':30,'test_samples':30,
            'solver':'lbfgs','alpha':0.01,'max_iter':3000})
        mlflow.log_metrics({k:float(v) for k,v in row.items() if k!='model'})
        if name in training_ms: mlflow.log_metric('training_time_ms',training_ms[name])
        mlflow.set_tags({'dataset':'sklearn Iris','dataset_sha256':dataset_hash,
            'python_version':platform.python_version(),'sklearn_version':sklearn.__version__,
            'mlflow_version':mlflow.__version__,'evaluation':'validation'})
        mlflow.log_dict(manifest,'data_split.json')
        mlflow.log_dict({'features':features,'units':'cm','classes':runtime.classes.tolist()},'input_schema.json')
        data = X.copy(); data['Species'] = y
        mlflow.log_text(data.to_csv(index=False),'Iris.csv')
        mlflow.log_artifact(str(output/(name+'.pkl')))
        info = mlflow.pyfunc.log_model(artifact_path='iris_model',python_model=IrisPythonModel(runtime),
            signature=infer_signature(X.iloc[train_idx],runtime.predict(X.iloc[train_idx])),
            input_example=X.head(2),
            pip_requirements=[f'numpy=={np.__version__}',f'pandas=={pd.__version__}',
                              f'cloudpickle=={cloudpickle.__version__}'])
    registered = mlflow.register_model(info.model_uri,'IrisNeuralDeployment')
    run_records[name] = {'run_id':run.info.run_id,'version':registered.version,'model_uri':info.model_uri}
    print(name,run_records[name])

ranked = comparison.sort_values(['validation_accuracy','model_size_bytes','prediction_p95_ms'],ascending=[False,True,True])
selected = ranked.iloc[0]['model']
version = run_records[selected]['version']
client.set_registered_model_alias('IrisNeuralDeployment','candidate',version)
current_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id],filter_string="attributes.status = 'FINISHED'")
current_runs = current_runs[current_runs.run_id.isin([r['run_id'] for r in run_records.values()])]
print(current_runs[['run_id','params.quantization','metrics.validation_accuracy','metrics.model_size_bytes']].to_string(index=False))
print('Selected:',selected,'| candidate version:',version)


/root/.local/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/10/15 13:21:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/15 13:21:59 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/10/15 13:21:59 INFO mlflow.pyfunc: Validating input example against model signature
Successfully registered model 'IrisNeuralDeployment'.
Created version '1' of m

## Task 4 Deployment recommendation
Use the smallest validated network that preserves the required accuracy. Quantise the compact MLP if validation supports it; otherwise retain its floating point form or the larger baseline if accuracy requires that. This notebook makes the choice using the stated validation ranking and evaluates it once on the untouched test set. A perfect result on 30 flowers would not establish real-world reliability.

Bundle preprocessing, feature order, class mapping and weights in one offline runtime. Pin package versions and retain the exact registered model version for audit and rollback. After acceptance on the intended device, ship the model with a form accepting four centimetre measurements. Validate numeric inputs and reject missing values; flag unsupported species and unusual conditions for expert review. Schedule model updates when connectivity is available. The measurements below establish this demonstration only; device latency, RAM, energy, maintenance and cost require separate field testing.

In [5]:
chosen = runtimes[selected]
loaded = mlflow.pyfunc.load_model(f'models:/IrisNeuralDeployment/{version}')
test_predictions = chosen.predict(X.iloc[test_idx])
assert np.array_equal(loaded.predict(X.iloc[test_idx]),test_predictions)
metrics = {'test_accuracy':accuracy_score(y[test_idx],test_predictions),
           'test_macro_f1':f1_score(y[test_idx],test_predictions,average='macro')}
with mlflow.start_run(run_id=run_records[selected]['run_id']): mlflow.log_metrics(metrics)
print('Selected model:',selected)
print('Untouched test results:',metrics)
print('Confusion matrix rows actual and columns predicted:',chosen.classes.tolist())
print(confusion_matrix(y[test_idx],test_predictions,labels=chosen.classes))
# cloudpickle persists the custom class so the offline file can load in a fresh process.
offline_path = output/'iris_deployment.pkl'
with offline_path.open('wb') as f: cloudpickle.dump(chosen,f)
with offline_path.open('rb') as f: offline = cloudpickle.load(f)
flower = pd.DataFrame([[5.1,3.5,1.4,0.2]],columns=features)
assert np.array_equal(offline.predict(X.iloc[test_idx]),test_predictions)
print('Offline example:',offline.predict(flower)[0])
summary = {'selected':selected,'version':str(version),'test_metrics':metrics,'runs':run_records,
           'comparison':rows,'features':features,'dataset_sha256':dataset_hash}
(output/'summary.json').write_text(json.dumps(summary,indent=2))
print('Evidence saved in:',output.resolve())


Selected model: compact_int8
Untouched test results: {'test_accuracy': 0.9, 'test_macro_f1': 0.8976982097186701}
Confusion matrix rows actual and columns predicted: ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
[[10  0  0]
 [ 0 10  0]
 [ 0  3  7]]
Offline example: Iris-setosa
Evidence saved in: /workspace/scratch/e982596c1aea/assignment27/iris_assignment_outputs


## Viewing the MLflow dashboard and submission
From a terminal in the notebook working directory run:

```bash
mlflow ui --backend-store-uri sqlite:///iris_assignment_outputs/iris_mlflow.db --port 5000
```

Open http://127.0.0.1:5000. Choose Assignment27_Iris_Neural_Deployment, select this execution’s three runs and compare their parameters and metrics. In Models, open IrisNeuralDeployment to inspect versions and the candidate alias. The written submission contains genuine execution logs as evidence, not fabricated dashboard screenshots.

For the template’s public notebook-link requirement, upload this executed notebook to Google Colab or your approved repository, set access to Anyone with the link can view, and paste that shareable URL into the submission. A local download is not a public sharing link. No public link is created by this notebook.

## References
- scikit-learn neural networks: https://scikit-learn.org/stable/modules/neural_networks_supervised.html
- ONNX Runtime quantisation and hardware considerations: https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html
- MLflow registry and aliases: https://mlflow.org/docs/latest/ml/model-registry/workflow/


## References
- scikit-learn neural networks: https://scikit-learn.org/stable/modules/neural_networks_supervised.html
- ONNX Runtime quantisation and hardware considerations: https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html
- MLflow registry and aliases: https://mlflow.org/docs/latest/ml/model-registry/workflow/
